# 03 · MNIST final training

Notebook này đọc thư mục dataset, train bốn model, lưu file weight và đánh giá trên validation/test.

In [ ]:
from pathlib import Path
import sys
import json
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'gk' / 'web' / 'backend').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from gk.web.backend.digits_models import build_logistic, build_mlp, export_layers, model_metadata

ARTIFACT_DIR = PROJECT_ROOT / 'gk' / 'web' / 'backend' / 'artifacts'
DATASET_DIR = ARTIFACT_DIR / 'mnist_dataset'
DATASET_PATH = DATASET_DIR / 'mnist_normalized.npz'
CONFIG_PATH = DATASET_DIR / 'mnist_model_config.json'
SPLIT_PATH = DATASET_DIR / 'split.json'
OUTPUT_PATH = DATASET_DIR / 'mnist_weights.json'
TRAIN_LOG_PATH = DATASET_DIR / 'training_log.json'
WEB_OUTPUT_PATH = ARTIFACT_DIR / 'digits_models.json'
data = np.load(DATASET_PATH)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8')) if CONFIG_PATH.exists() else None
X_train_images = data['X_train_pixels'].astype(float)
y_train = data['y_train'].astype(int)
X_test_images = data['X_test_pixels'].astype(float)
y_test = data['y_test'].astype(int)
X_train_pixels = X_train_images.reshape(len(X_train_images), -1)
X_test_pixels = X_test_images.reshape(len(X_test_images), -1)
split = json.loads(SPLIT_PATH.read_text(encoding='utf-8'))
fit_indices = np.asarray(split['fit_indices'], dtype=int)
validation_indices = np.asarray(split['validation_indices'], dtype=int)
if config is None:
    scaler = StandardScaler().fit(X_train_pixels[fit_indices])
    config = {
        'version': 1, 'random_seed': split['random_seed'], 'dataset_file': DATASET_PATH.name,
        'input_shape': [8, 8], 'feature_count': 64, 'class_count': 10,
        'fit_indices': fit_indices.tolist(), 'validation_indices': validation_indices.tolist(),
        'test_indices': np.arange(len(y_test)).astype(int).tolist(),
        'scaler': {'mean': scaler.mean_.tolist(), 'std': scaler.scale_.tolist()},
        'models': {
            'logistic': {'kind': 'linear', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 2000},
            'mlp_4_one_layer': {'kind': 'ann', 'hidden_layer_sizes': [4], 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False},
            'mlp_4_two_layers': {'kind': 'ann', 'hidden_layer_sizes': [4, 4], 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False},
            'compact_tuned': {'kind': 'ann', 'hidden_layer_sizes': [48], 'activation': 'relu', 'learning_rate_init': 0.002, 'max_iter': 350, 'early_stopping': False},
        },
        'search': {'sample_count': 0, 'logistic_rounds': 0, 'mlp_rounds': 0, 'logistic_candidates': [], 'mlp_candidates': [], 'selected_logistic_validation_accuracy': None, 'selected_mlp_validation_accuracy': None},
    }
    CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
mean = np.asarray(config['scaler']['mean'], dtype=float)
std = np.asarray(config['scaler']['std'], dtype=float)
X_all = (X_train_pixels - mean) / std
X_test = (X_test_pixels - mean) / std
X_fit, y_fit = X_all[fit_indices], y_train[fit_indices]
X_validation, y_validation = X_all[validation_indices], y_train[validation_indices]
print('Loaded dataset and model config')

## Train bốn model và ghi log

Các config ở đây đến từ `mnist_model_config.json`, vì vậy có thể tối ưu notebook model mà không sửa notebook train. Log bên dưới lấy trực tiếp từ object đã fit.

In [ ]:
model_configs = config['models']
trained = {}
training_log = []

def print_mlp_loss_log(model_id, loss_curve):
    for epoch, loss in enumerate(loss_curve, start=1):
        if epoch == 1 or epoch % 10 == 0 or epoch == len(loss_curve):
            print(f'    epoch {epoch:>3}: loss={loss:.6f}')

def fit_and_log(model_id, model, model_config, kind):
    print(f'[{len(training_log) + 1}/4] {model_id} · start')
    print('    config:', model_config)
    started = perf_counter()
    model.fit(X_fit, y_fit)
    elapsed = perf_counter() - started
    loss_curve = [float(value) for value in getattr(model, 'loss_curve_', [])]
    if loss_curve:
        print_mlp_loss_log(model_id, loss_curve)
    iterations = int(np.max(model.n_iter_)) if hasattr(model, 'n_iter_') else None
    record = {
        'model_id': model_id, 'kind': kind, 'status': 'completed',
        'elapsed_seconds': round(elapsed, 3),
        'iterations': iterations, 'epochs': len(loss_curve) or None,
        'initial_loss': loss_curve[0] if loss_curve else None,
        'final_loss': loss_curve[-1] if loss_curve else None,
        'loss_curve': loss_curve,
        'config': model_config,
    }
    training_log.append(record)
    print(f'    done in {elapsed:.2f}s · iterations={iterations} · epochs={record["epochs"]} · final_loss={record["final_loss"]}')
    return model

logistic_config = model_configs['logistic']
trained['logistic'] = fit_and_log(
    'logistic',
    build_logistic(logistic_config['C'], logistic_config.get('solver', 'lbfgs'), logistic_config.get('max_iter', 2000), verbose=1),
    logistic_config, 'linear',
)

for model_id in ('mlp_4_one_layer', 'mlp_4_two_layers', 'compact_tuned'):
    model_config = model_configs[model_id]
    trained[model_id] = fit_and_log(
        model_id,
        build_mlp(
            tuple(model_config['hidden_layer_sizes']), model_config['activation'],
            model_config['learning_rate_init'], model_config['max_iter'],
            model_config.get('early_stopping', False), model_config.get('n_iter_no_change'), verbose=True,
        ),
        model_config, 'ann',
    )

print('trained:', list(trained))

In [ ]:
results = {}
for model_id, model in trained.items():
    results[model_id] = {
        'validation_accuracy': accuracy_score(y_validation, model.predict(X_validation)),
        'test_accuracy': accuracy_score(y_test, model.predict(X_test)),
    }
    record = next(item for item in training_log if item['model_id'] == model_id)
    record.update(results[model_id])
    print(f'{model_id:24} validation={results[model_id]["validation_accuracy"]:.2%} test={results[model_id]["test_accuracy"]:.2%} elapsed={record["elapsed_seconds"]:.2f}s')
TRAIN_LOG_PATH.write_text(json.dumps({
    'dataset': str(DATASET_PATH), 'train_samples': len(X_fit),
    'validation_samples': len(X_validation), 'test_samples': len(X_test),
    'models': training_log,
}, indent=2), encoding='utf-8')
print('saved training log:', TRAIN_LOG_PATH)

## Confusion matrix và loss curve

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for axis, (model_id, model) in zip(axes, trained.items()):
    matrix = confusion_matrix(y_test, model.predict(X_test), labels=np.arange(10))
    axis.imshow(matrix, cmap='Blues')
    axis.set_title(model_id)
    axis.set_xlabel('predicted')
    axis.set_ylabel('true')
plt.tight_layout()

plt.figure(figsize=(6, 3))
for model_id in ('mlp_4_one_layer', 'mlp_4_two_layers', 'compact_tuned'):
    model = trained[model_id]
    plt.plot(model.loss_curve_, label=model_id)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.tight_layout()

In [ ]:
sample_index = 7
plt.figure(figsize=(2, 2))
plt.imshow(X_test_pixels[sample_index].reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
plt.title(f'true label: {y_test[sample_index]}')
plt.axis('off')
plt.show()
for model_id, model in trained.items():
    probabilities = model.predict_proba(X_test[sample_index:sample_index + 1])[0]
    print(model_id, '→', int(np.argmax(probabilities)), f'{probabilities.max():.2%}')

## Export artifact cho web

In [ ]:
baseline_predictions = trained['logistic'].predict(X_test)
tuned_predictions = trained['compact_tuned'].predict(X_test)
demo_sample_index = 0
for index, baseline_prediction, tuned_prediction, target in zip(
    range(len(y_test)), baseline_predictions, tuned_predictions, y_test
):
    if baseline_prediction != target and tuned_prediction == target:
        demo_sample_index = int(index)
        break

def make_metadata(model_id, result_key, name, kind, model, architecture, activations, hidden_activation, source=None, source_url=None):
    return model_metadata(
        model_id, name, kind, model, architecture, activations,
        results[result_key]['test_accuracy'],
        results[result_key]['validation_accuracy'],
        hidden_activation, source, source_url,
    )

compact_config = model_configs['compact_tuned']
models = [
    make_metadata('logistic', 'logistic', 'Logistic Regression', 'linear', trained['logistic'], [64, 10], [], None, 'scikit-learn LogisticRegression', 'https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html'),
    make_metadata('mlp-4-one-layer', 'mlp_4_one_layer', 'MLP · 4 neurons · 1 layer', 'ann', trained['mlp_4_one_layer'], [64, 4, 10], ['tanh', 'softmax'], 'tanh'),
    make_metadata('mlp-4-two-layer', 'mlp_4_two_layers', 'MLP · 4 neurons · 2 layers', 'ann', trained['mlp_4_two_layers'], [64, 4, 4, 10], ['tanh', 'tanh', 'softmax'], 'tanh'),
    make_metadata('compact-tuned', 'compact_tuned', 'Compact tuned MLP', 'ann', trained['compact_tuned'], [64, *compact_config['hidden_layer_sizes'], 10], [compact_config['activation']] * len(compact_config['hidden_layer_sizes']) + ['softmax'], compact_config['activation']),
]

artifact = {
    'version': 3,
    'random_seed': config['random_seed'],
    'dataset': {
        'name': 'MNIST handwritten digits',
        'description': 'Real handwritten digit images from MNIST, normalized to 8x8 grayscale pixels for this demo.',
        'sample_count': 70000, 'training_sample_count': 60000, 'test_sample_count': 10000,
        'input_shape': [8, 8], 'feature_count': 64, 'class_count': 10,
        'pixel_min': 0, 'pixel_max': 16, 'original_input_shape': [28, 28],
        'source_url': 'https://yann.lecun.com/exdb/mnist/',
    },
    'preprocessing': {
        'name': 'CropSquareResize8 + StandardScaler',
        'feature_transform': 'denoise at 50% of max ink, keep largest component, crop foreground, resize to square, area-average resize to 8x8, scale intensity to 0..16',
        'mean': mean.tolist(), 'std': std.tolist(),
    },
    'fit_indices': config['fit_indices'], 'validation_indices': config['validation_indices'],
    'test_indices': config['test_indices'],
    'test_samples': np.rint(X_test_pixels).astype(int).tolist(),
    'test_labels': y_test.astype(int).tolist(),
    'demo_sample_index': demo_sample_index,
    'augmentation': {'source_count': 48000, 'generated_count': 48000, 'shifts': [], 'rotations_degrees': [], 'applied_to': 'disabled for MNIST; the source already contains real handwritten training images'},
    'baseline_tuning': {'rounds': config['search']['logistic_rounds'], 'C_candidates': [row['C'] for row in config['search']['logistic_candidates']], 'selected_C': model_configs['logistic']['C'], 'validation_accuracy': config['search']['selected_logistic_validation_accuracy']},
    'tuning': {'rounds': config['search']['mlp_rounds'], 'search_sample_count': config['search']['sample_count'], 'solver': 'adam', 'search_max_iter': 120, 'final_max_iter': compact_config['max_iter'], 'early_stopping': compact_config['early_stopping'], 'candidates': config['search']['mlp_candidates'], 'selected': compact_config['hidden_layer_sizes'], 'selected_activation': compact_config['activation'], 'selected_learning_rate_init': compact_config['learning_rate_init'], 'validation_accuracy': config['search']['selected_mlp_validation_accuracy']},
    'models': models,
}
serialized_artifact = json.dumps(artifact, indent=2)
OUTPUT_PATH.write_text(serialized_artifact, encoding='utf-8')
WEB_OUTPUT_PATH.write_text(serialized_artifact, encoding='utf-8')
print(f'Wrote weights: {OUTPUT_PATH}')
print(f'Updated web artifact: {WEB_OUTPUT_PATH}')